# Assignment 1 — Gradio dashboard

## 1. Setup

In [1]:
import json
from pathlib import Path

import gradio as gr
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import torch.nn as nn

ARTIFACTS_DIR = Path("artifacts")

/Users/joseluishervasdecela_1/daiist-assignment-1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open(ARTIFACTS_DIR / "metadata.json") as f:
    meta = json.load(f)

FEATURE_NAMES = meta["feature_names"]
COST_FN = meta["cost_fn"]
COST_FP = meta["cost_fp"]
CHOSEN_THRESHOLD = meta["chosen_threshold"]

data = pd.read_csv(ARTIFACTS_DIR / "dashboard_data.csv", parse_dates=["created"])
test_results = pd.read_csv(ARTIFACTS_DIR / "test_results.csv")
ablation = pd.read_csv(ARTIFACTS_DIR / "ablation.csv")
print(data.shape, "| features:", len(FEATURE_NAMES))

(3395, 23) | features: 15


## 2. Load the three models

In [3]:
scaler = joblib.load(ARTIFACTS_DIR / "scaler.joblib")
sk_model = joblib.load(ARTIFACTS_DIR / "sklearn_model.joblib")

manual_params = torch.load(ARTIFACTS_DIR / "manual_torch_model.pt")
w_manual, b_manual = manual_params["w"], manual_params["b"]

std_model = nn.Sequential(nn.Linear(len(FEATURE_NAMES), 1), nn.Sigmoid())
std_model.load_state_dict(torch.load(ARTIFACTS_DIR / "standard_torch_model.pt"))
std_model.eval()

Sequential(
  (0): Linear(in_features=15, out_features=1, bias=True)
  (1): Sigmoid()
)

## 3. Predictions for every session

I scale the features with the saved scaler (the same one fitted on train) and get the probability of a long session from each model. This is inference only.

In [4]:
X = scaler.transform(data[FEATURE_NAMES].to_numpy(dtype=float))
X_t = torch.tensor(X, dtype=torch.float32)

with torch.no_grad():
    data["p_sklearn"] = sk_model.predict_proba(X)[:, 1]
    data["p_manual_torch"] = torch.sigmoid(X_t @ w_manual + b_manual).numpy()
    data["p_standard_torch"] = std_model(X_t).squeeze(-1).numpy()

MODELS = ["sklearn", "manual_torch", "standard_torch"]
SPLIT_LABELS = {"Test": "test", "Validation": "val", "Train": "train"}
# Fixed order and colours so "short" and "long" look the same in every chart
OUTCOME_ORDER = ["short (≤ 4 h)", "long (> 4 h)"]
OUTCOME_COLORS = {"short (≤ 4 h)": "#4C78A8", "long (> 4 h)": "#E45756"}
print(data[["target"] + [f"p_{m}" for m in MODELS]].describe().round(3))


         target  p_sklearn  p_manual_torch  p_standard_torch
count  3395.000   3395.000        3395.000          3395.000
mean      0.112      0.115           0.116             0.116
std       0.315      0.110           0.110             0.110
min       0.000      0.001           0.001             0.001
25%       0.000      0.050           0.050             0.050
50%       0.000      0.088           0.088             0.088
75%       0.000      0.136           0.136             0.136
max       1.000      0.942           0.942             0.942


## 4. Helper functions

In [5]:
def confusion_counts(y_true, probs, threshold):
    alert = probs >= threshold
    tp = int((alert & (y_true == 1)).sum())
    fp = int((alert & (y_true == 0)).sum())
    fn = int((~alert & (y_true == 1)).sum())
    tn = int((~alert & (y_true == 0)).sum())
    return tp, fp, fn, tn

def business_cost(y_true, probs, threshold, cost_fn=COST_FN, cost_fp=COST_FP):
    tp, fp, fn, tn = confusion_counts(y_true, probs, threshold)
    return cost_fn * fn + cost_fp * fp

In [6]:
import plotly.io as pio

# Palette: asphalt greys, the teal of a charging light, amber for a blocked charger
COLORS = {
    "ink": "#1C2B39",
    "muted": "#5E6E7E",
    "canvas": "#F2F4F3",
    "line": "#D8DEDC",
    "charge": "#0F766E",
    "short": "#7D93AE",
    "long": "#D98C1F",
}
MODEL_LABELS = {"sklearn": "scikit-learn", "manual_torch": "Manual PyTorch",
                "standard_torch": "Standard PyTorch"}
MODEL_COLORS = {"sklearn": "#0F766E", "manual_torch": "#24476B", "standard_torch": "#8FB3C9"}

# Same order and colours for short and long sessions in every chart
OUTCOME_ORDER = ["short (≤ 4 h)", "long (> 4 h)"]
OUTCOME_COLORS = {"short (≤ 4 h)": COLORS["short"], "long (> 4 h)": COLORS["long"]}

# One Plotly look for every chart, so each function doesn't have to style its own figure
pio.templates["charging"] = go.layout.Template(layout=dict(
    font=dict(family="IBM Plex Sans, Helvetica, Arial, sans-serif", color=COLORS["ink"], size=13),
    title=dict(font=dict(size=15), x=0, xref="paper", xanchor="left"),
    paper_bgcolor="white", plot_bgcolor="white",
    xaxis=dict(gridcolor="#EDF0EF", linecolor=COLORS["line"], zeroline=False),
    yaxis=dict(gridcolor="#EDF0EF", linecolor=COLORS["line"], zeroline=False),
    legend=dict(title_text="", orientation="h", yanchor="top", y=-0.22, xanchor="left", x=0),
    margin=dict(l=60, r=20, t=60, b=40),
))
pio.templates.default = "plotly_white+charging"

THEME = gr.themes.Base(
    primary_hue=gr.themes.colors.teal,
    neutral_hue=gr.themes.colors.slate,
    font=[gr.themes.GoogleFont("IBM Plex Sans"), "Helvetica", "Arial", "sans-serif"],
    radius_size=gr.themes.sizes.radius_sm,
).set(
    body_background_fill=COLORS["canvas"],
    block_border_color=COLORS["line"],
    block_shadow="none",
)

CSS = """
.gradio-container { width: 100% !important; max-width: 1180px !important; margin: 0 auto !important; }
#header { padding: 8px 0 4px; }
#header p { color: #5E6E7E; }
#header h1 { font-size: 30px; font-weight: 600; letter-spacing: -0.01em; margin: 0 0 8px; }
#header p { max-width: 72ch; margin: 0 0 4px; }
.result-panel { padding: 4px 2px 8px; }
.cost-line { display: flex; align-items: baseline; gap: 14px; flex-wrap: wrap; margin-bottom: 10px; }
.cost-value { font-size: 44px; font-weight: 600; line-height: 1; }
.cost-context { color: var(--body-text-color-subdued); }
.cost-track { display: flex; height: 14px; background: #E3E8E6; border-radius: 2px; overflow: hidden; }
.seg-missed { background: #D98C1F; }
.seg-reminders { background: #7D93AE; }
.cost-legend { display: flex; gap: 20px; flex-wrap: wrap; margin-top: 8px; font-size: 13px;
               color: var(--body-text-color-subdued); }
.swatch { display: inline-block; width: 10px; height: 10px; margin-right: 6px; border-radius: 2px; }
.figures { display: grid; grid-template-columns: repeat(3, 1fr); margin-top: 22px;
           border-top: 1px solid var(--border-color-primary); }
.figure { padding: 14px 18px 2px 0; }
.figure + .figure { padding-left: 18px; border-left: 1px solid var(--border-color-primary); }
.figure .value { font-size: 24px; font-weight: 600; }
.figure .label { font-size: 13px; color: var(--body-text-color-subdued); margin-top: 2px; }
@media (max-width: 700px) {
  .figures { grid-template-columns: 1fr; }
  .figure + .figure { padding-left: 0; border-left: none; border-top: 1px solid var(--border-color-primary); }
}
"""

def result_panel(tp, fp, fn, tn, cost_fn, cost_fp):
    """HTML summary of one threshold: the cost against reminding nobody, then three key figures."""
    missed_cost, reminder_cost = fn * cost_fn, fp * cost_fp
    cost = missed_cost + reminder_cost
    cost_nobody = (tp + fn) * cost_fn
    scale = max(cost, cost_nobody, 1)
    saving = cost_nobody - cost
    recall = tp / (tp + fn) if tp + fn else 0.0
    precision = tp / (tp + fp) if tp + fp else 0.0
    bar_note = ("Full bar: cost of reminding nobody" if cost <= cost_nobody
                else "Full bar: this threshold's cost")
    saving_text = (f"{saving} less than reminding nobody ({cost_nobody})" if saving >= 0
                   else f"{-saving} more than reminding nobody ({cost_nobody})")
    return f"""
    <div class="result-panel">
      <div class="cost-line">
        <span class="cost-value">{cost}</span>
        <span class="cost-context">business cost, {saving_text}</span>
      </div>
      <div class="cost-track" role="img"
           aria-label="Cost {cost}: missed sessions {missed_cost}, unnecessary reminders {reminder_cost}">
        <div class="seg-missed" style="width:{missed_cost / scale:.1%}"></div>
        <div class="seg-reminders" style="width:{reminder_cost / scale:.1%}"></div>
      </div>
      <div class="cost-legend">
        <span><span class="swatch" style="background:#D98C1F"></span>{fn} missed long sessions × {cost_fn} = {missed_cost}</span>
        <span><span class="swatch" style="background:#7D93AE"></span>{fp} unnecessary reminders × {cost_fp} = {reminder_cost}</span>
        <span>{bar_note}</span>
      </div>
      <div class="figures">
        <div class="figure"><div class="value">{tp} of {tp + fn}</div>
          <div class="label">long sessions caught (recall {recall:.0%})</div></div>
        <div class="figure"><div class="value">{tp + fp}</div>
          <div class="label">reminders sent ({precision:.0%} of them justified)</div></div>
        <div class="figure"><div class="value">{tn}</div>
          <div class="label">short sessions correctly left alone</div></div>
      </div>
    </div>"""

## 4. Tab Compare models: prediction vs. actual

For a classifier, "prediction vs. actual" means: when a model says 30%, do about 30% of those sessions really turn out long? I sort the sessions into 10 groups by predicted probability and plot the average prediction against the real share of long sessions. If the three models are really the same, their lines should sit on top of each other. The second plot shows the predicted probabilities split by what actually happened: long sessions should get higher probabilities.

In [7]:
def calibration_plot(split_label):
    d = data[data["split"] == SPLIT_LABELS[split_label]]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=[0, 0.7], y=[0, 0.7], mode="lines", name="perfect calibration",
                             line=dict(dash="dot", color=COLORS["muted"], width=1)))
    for m, dash in zip(MODELS, ["solid", "dash", "dot"]):
        bins = pd.qcut(d[f"p_{m}"], q=10, duplicates="drop")
        g = d.groupby(bins, observed=True).agg(pred=(f"p_{m}", "mean"), actual=("target", "mean"))
        fig.add_trace(go.Scatter(x=g["pred"], y=g["actual"], mode="lines+markers", name=MODEL_LABELS[m],
                                 line=dict(dash=dash, color=MODEL_COLORS[m], width=2)))
    fig.update_layout(title=f"Predicted vs. actual share of long sessions ({split_label})",
                      xaxis_title="average predicted probability",
                      yaxis_title="actual share of sessions > 4 h", height=460)
    return fig

def probability_boxplot(split_label):
    d = data[data["split"] == SPLIT_LABELS[split_label]]
    long_df = d.melt(id_vars="target", value_vars=[f"p_{m}" for m in MODELS],
                     var_name="model", value_name="predicted probability")
    long_df["model"] = long_df["model"].str.replace("p_", "", regex=False).map(MODEL_LABELS)
    long_df["what happened"] = long_df["target"].map({0: "short (≤ 4 h)", 1: "long (> 4 h)"})
    fig = px.box(long_df, x="model", y="predicted probability", color="what happened",
                 category_orders={"what happened": OUTCOME_ORDER}, color_discrete_map=OUTCOME_COLORS,
                 title=f"Predicted probability by what actually happened ({split_label})")
    fig.update_layout(height=460, xaxis_title=None, legend_title_text=None)
    return fig

def max_difference_text(split_label):
    d = data[data["split"] == SPLIT_LABELS[split_label]]
    diff_manual = np.abs(d["p_manual_torch"] - d["p_sklearn"]).max()
    diff_std = np.abs(d["p_standard_torch"] - d["p_sklearn"]).max()
    return (f"The three models agree: on the {split_label.lower()} set, no predicted probability differs "
            f"from scikit-learn's by more than **{max(diff_manual, diff_std):.4f}** "
            f"(manual PyTorch {diff_manual:.4f}, standard PyTorch {diff_std:.4f}).")

## 6. Tab Data: feature and target distributions

Pick a column and a split to see how it's distributed, coloured by whether the session ended up long or short. Percentages are within each colour, so long and short sessions can be compared even though long ones are rare. There's also a small chart with the share of long sessions in each split.

In [8]:
# Readable names for the columns you can pick
DISTRIBUTION_COLUMNS = {
    "chargeTimeHrs": "Session duration (hours)",
    "startTime": "Start hour",
    "weekday": "Weekday",
    "facilityType": "Facility type",
    "distance_raw": "Commute distance (original, with gaps)",
    "distance_missing": "Distance missing",
    "manager_vehicle": "Manager vehicle",
    "reported_zip": "Reported ZIP code",
    "user_prev_long_share": "Driver's share of past long sessions",
    "user_prev_mean_hours": "Driver's average past duration (hours)",
    "user_prev_sessions": "Driver's number of past sessions (log)",
}
WEEKDAY_ORDER = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

def distribution_plot(column, split_label):
    d = data if split_label == "All" else data[data["split"] == SPLIT_LABELS[split_label]]
    d = d.assign(**{"what happened": d["target"].map({0: "short (≤ 4 h)", 1: "long (> 4 h)"})})
    label = DISTRIBUTION_COLUMNS[column]
    fig = px.histogram(d, x=column, color="what happened", barmode="overlay",
                       histnorm="percent", opacity=0.7,
                       category_orders={"what happened": OUTCOME_ORDER, "weekday": WEEKDAY_ORDER},
                       color_discrete_map=OUTCOME_COLORS,
                       title=f"{label}: short vs. long sessions ({split_label.lower()} data)")
    if column == "chargeTimeHrs":
        fig.add_vline(x=meta["long_session_hours"], line_dash="dash", line_color=COLORS["ink"],
                      annotation_text="4-hour limit", annotation_position="top left")
        fig.update_xaxes(range=[0, 12])   # a few sessions last 50+ hours and squash the rest
    fig.update_layout(height=460, xaxis_title=label, yaxis_title="% of sessions in each group",
                      legend_title_text=None)
    return fig

def target_by_split_plot():
    g = data.groupby("split", sort=False)["target"].mean().reindex(["train", "val", "test"])
    fig = go.Figure(go.Bar(x=["Train", "Validation", "Test"], y=g.values,
                           text=[f"{v:.1%}" for v in g.values], textposition="outside",
                           marker_color=COLORS["long"], width=0.5))
    fig.update_layout(title="Share of long sessions in each split", yaxis_tickformat=".0%",
                      yaxis_range=[0, max(g.values) * 1.3], height=340, showlegend=False)
    return fig

## 7. Tab "Threshold & cost"

This is the business part. Move the threshold and see what happens: how many blocking sessions are caught, how many reminders go out, and what it all costs compared with reminding nobody. You can also change how much each error costs, in case the company sees it differently than I do. The bar shows where the cost comes from: missed sessions in amber, unnecessary reminders in blue.

In [9]:
def threshold_view(model, split_label, threshold, cost_fn, cost_fp):
    cost_fn, cost_fp = cost_fn or 0, cost_fp or 0   # an empty box counts as 0
    d = data[data["split"] == SPLIT_LABELS[split_label]]
    y, p = d["target"].to_numpy(), d[f"p_{model}"].to_numpy()
    tp, fp, fn, tn = confusion_counts(y, p, threshold)
    cost = business_cost(y, p, threshold, cost_fn, cost_fp)

    cm = go.Figure(go.Heatmap(
        z=[[tn, fp], [fn, tp]],
        x=["no reminder", "reminder"], y=["short (≤ 4 h)", "long (> 4 h)"],
        text=[[f"<b>{tn}</b><br>correctly left alone", f"<b>{fp}</b><br>unnecessary reminders"],
              [f"<b>{fn}</b><br>missed: a colleague<br>can't charge", f"<b>{tp}</b><br>caught"]],
        texttemplate="%{text}", textfont=dict(size=14),
        colorscale=[[0, "#F4F7F6"], [1, "#9BCFC9"]], showscale=False, xgap=3, ygap=3))
    cm.update_layout(title=f"Confusion matrix at threshold {threshold:.2f}",
                     xaxis_title="model decision", yaxis_title="what actually happened",
                     yaxis_autorange="reversed", height=400)

    grid = np.round(np.arange(0.01, 0.91, 0.01), 2)
    costs = [business_cost(y, p, t, cost_fn, cost_fp) for t in grid]
    curve = go.Figure(go.Scatter(x=grid, y=costs, mode="lines", name="cost",
                                 line=dict(color=COLORS["charge"], width=2.5)))
    curve.add_trace(go.Scatter(x=[threshold], y=[cost], mode="markers", name="current threshold",
                               marker=dict(size=12, color=COLORS["long"],
                                           line=dict(color="white", width=2))))
    curve.add_vline(x=CHOSEN_THRESHOLD[model], line_dash="dash", line_color=COLORS["muted"],
                    annotation_text="chosen on validation", annotation_position="top right")
    curve.update_layout(title="Business cost for every threshold", xaxis_title="threshold",
                        yaxis_title="business cost", height=400, showlegend=False)
    return cm, result_panel(tp, fp, fn, tn, cost_fn, cost_fp), curve

In [10]:
# Tables with readable names for the overview tab
overview_results = test_results.rename(columns={
    "model": "Model", "log_loss": "Log loss", "roc_auc": "ROC AUC", "pr_auc": "PR AUC",
    "threshold": "Threshold", "recall": "Recall", "precision": "Precision",
    "business_cost": "Business cost"})
overview_results["Model"] = overview_results["Model"].replace(
    {"naive baseline": "Naive baseline", **MODEL_LABELS})
overview_ablation = ablation.rename(columns={
    "removed": "Feature group removed", "val_pr_auc": "PR AUC (validation)",
    "val_best_cost": "Business cost (validation)"})

def overview_panel():
    d = data[data["split"] == "test"]
    y, p = d["target"].to_numpy(), d["p_sklearn"].to_numpy()
    return result_panel(*confusion_counts(y, p, CHOSEN_THRESHOLD["sklearn"]), COST_FN, COST_FP)

MODEL_CHOICES = [(MODEL_LABELS[m], m) for m in MODELS]
COLUMN_CHOICES = [(label, col) for col, label in DISTRIBUTION_COLUMNS.items()]

with gr.Blocks(title="EV charger blocking: dashboard") as demo:
    gr.HTML(
        "<h1>Will this charging session block the charger?</h1>"
        f"<p>When an employee plugs in, the model estimates whether the car will stay more than "
        f"{meta['long_session_hours']} hours, so the company can send a reminder before a colleague is left without a charger.</p>"
        "<p>The same logistic regression is trained three ways. Everything shown here is loaded from "
        "<code>artifacts/</code>; nothing is retrained.</p>",
        elem_id="header",
    )

    with gr.Tab("Overview"):
        gr.Markdown(f"### Test month, threshold {CHOSEN_THRESHOLD['sklearn']:.2f} (chosen on validation)")
        overview_html = gr.HTML()
        gr.Markdown(f"### All models on the test set\nA missed long session costs {COST_FN}, "
                    f"an unnecessary reminder costs {COST_FP}.")
        gr.Dataframe(overview_results.round(3), interactive=False)
        gr.Markdown("### What each feature group adds\nThe model is retrained without one group at a time "
                    "and scored on validation. A higher cost without a group means the group was helping.")
        gr.Dataframe(overview_ablation.round(3), interactive=False)

    with gr.Tab("Compare models"):
        split_cmp = gr.Radio(["Test", "Validation", "Train"], value="Test", label="Data")
        diff_md = gr.Markdown()
        with gr.Row():
            calib = gr.Plot(show_label=False)
            box = gr.Plot(show_label=False)
        split_cmp.change(calibration_plot, split_cmp, calib)
        split_cmp.change(probability_boxplot, split_cmp, box)
        split_cmp.change(max_difference_text, split_cmp, diff_md)

    with gr.Tab("Data"):
        with gr.Row():
            col_dd = gr.Dropdown(COLUMN_CHOICES, value="chargeTimeHrs", label="Column", scale=2)
            split_dist = gr.Radio(["All", "Train", "Validation", "Test"], value="All", label="Data", scale=3)
        dist = gr.Plot(show_label=False)
        target_plot = gr.Plot(show_label=False)
        col_dd.change(distribution_plot, [col_dd, split_dist], dist)
        split_dist.change(distribution_plot, [col_dd, split_dist], dist)

    with gr.Tab("Threshold & cost"):
        with gr.Row():
            model_dd = gr.Dropdown(MODEL_CHOICES, value="sklearn", label="Model")
            split_thr = gr.Radio(["Test", "Validation"], value="Test", label="Data")
            cfn = gr.Number(value=COST_FN, label="Cost of a missed long session")
            cfp = gr.Number(value=COST_FP, label="Cost of an unnecessary reminder")
        thr = gr.Slider(0.01, 0.90, value=CHOSEN_THRESHOLD["sklearn"], step=0.01,
                        label="Send a reminder when the predicted probability is at least")
        result_html = gr.HTML()
        with gr.Row():
            cm_plot = gr.Plot(show_label=False)
            curve_plot = gr.Plot(show_label=False)
        inputs = [model_dd, split_thr, thr, cfn, cfp]
        outputs = [cm_plot, result_html, curve_plot]
        for control in inputs:
            control.change(threshold_view, inputs, outputs)

    def draw_all():
        return (overview_panel(),
                calibration_plot("Test"), probability_boxplot("Test"), max_difference_text("Test"),
                distribution_plot("chargeTimeHrs", "All"), target_by_split_plot(),
                *threshold_view("sklearn", "Test", CHOSEN_THRESHOLD["sklearn"], COST_FN, COST_FP))

    demo.load(draw_all, None, [overview_html, calib, box, diff_md, dist, target_plot,
                               cm_plot, result_html, curve_plot])

In [ ]:
demo.launch(theme=THEME, css=CSS)